## Lab 2 — OOP Refactor & FastAPI CRUD API


## 0 — Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'fastapi', 'uvicorn[standard]', 'httpx', 'pydantic', '-q'])
print('✅ Dependencies installed')

---
## Task 1 — Model the Server with a Dataclass

### 📖 Concept: `@dataclass`

A `dataclass` auto-generates `__init__`, `__repr__`, and `__eq__` from field annotations. Use it for data containers that don't need complex logic.

```python
from dataclasses import dataclass, field

@dataclass
class Config:
    host: str
    port: int = 8080
    tags: list[str] = field(default_factory=list)  # mutable defaults need field()

cfg = Config('localhost')
print(cfg)  # Config(host='localhost', port=8080, tags=[])
```

> Never use `tags: list = []` as a default — that creates one shared list for all instances. Always use `field(default_factory=list)`.

In [ ]:
from dataclasses import dataclass, field

# ✏️ YOUR TURN
# Create a Server dataclass with fields:
#   id: int
#   name: str
#   host: str
#   port: int
#   status: str = 'unknown'
#   tags: list[str] = (use field(default_factory=list))
#
# Add a method base_url(self) -> str that returns 'http://{host}:{port}'

# TODO: implement Server dataclass


# Test it
s = Server(id=1, name='api-prod-1', host='10.0.0.1', port=8080)
print(s)
print(s.base_url())

---
## Task 2 — ConfigLoader Class

### 📖 Concept: Classes with logging

A class groups related data (the file path) and behaviour (the `load()` method). Use `logging` instead of `print()` for anything that goes to production.

```python
import logging
logger = logging.getLogger(__name__)
logger.info('Loading %d items', count)   # use %s placeholders, not f-strings
```

In [ ]:
import json
import logging
import pathlib

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)


class ConfigError(ValueError):
    """Raised when config loading fails."""
    pass


# ✏️ YOUR TURN
# Implement ConfigLoader:
#   __init__(self, path: str) — store path as pathlib.Path
#   load(self) -> list[Server] — parse JSON, return list of Server objects
#   Raise ConfigError on FileNotFoundError or JSONDecodeError
#   Log info on success

class ConfigLoader:
    """Loads server configuration from a JSON file."""

    def __init__(self, path: str):
        # TODO: implement
        pass

    def load(self) -> list:
        """Parse the config file and return Server objects."""
        # TODO: implement
        pass


# Test it
loader = ConfigLoader('servers.json')
server_objects = loader.load()
print(server_objects)

---
## Task 3 — HealthChecker Class (async)

### 📖 Concept: `async`/`await` and `asyncio.gather()`

An `async def` function is a coroutine — it can pause and yield control while waiting for I/O (network, disk). This lets Python handle other work in the meantime.

`asyncio.gather()` runs multiple coroutines **concurrently** — instead of checking 10 servers one by one (10 × 1s = 10s), it checks all 10 at once (~1s total).

```python
import asyncio, httpx

async def fetch(url: str) -> int:
    async with httpx.AsyncClient() as client:
        resp = await client.get(url)
        return resp.status_code

# Run one coroutine
code = asyncio.run(fetch('https://example.com'))

# Run many concurrently
codes = asyncio.run(asyncio.gather(fetch('https://a.com'), fetch('https://b.com')))
```

In [ ]:
import asyncio
import time
import httpx

# ✏️ YOUR TURN
# Implement HealthChecker:
#   __init__(self, timeout=5.0, degraded_threshold_ms=500.0)
#   async check(self, server: Server) -> Server
#     — GET {server.base_url()}/health with httpx.AsyncClient
#     — Set server.status to UP / DEGRADED / DOWN
#     — Return the server
#   async check_all(self, servers: list[Server]) -> list[Server]
#     — Use asyncio.gather() to check all servers concurrently

class HealthChecker:
    """Checks server health over HTTP asynchronously."""

    def __init__(self, timeout: float = 5.0, degraded_threshold_ms: float = 500.0):
        # TODO: implement
        pass

    async def check(self, server) -> object:
        # TODO: implement
        pass

    async def check_all(self, servers: list) -> list:
        # TODO: implement
        pass


# Quick test with a real server
checker = HealthChecker()
test_server = Server(id=99, name='httpbin', host='httpbin.org', port=443)
# We can't use asyncio.run() inside Jupyter — use await directly:
result = await checker.check(test_server)
print(result)

---
## Task 4 — FastAPI CRUD API

### 📖 Concept: Pydantic models for validation

FastAPI uses Pydantic models to validate incoming request bodies and serialise responses automatically.

```python
from pydantic import BaseModel, Field

class ServerIn(BaseModel):
    name: str
    port: int = Field(default=8080, ge=1, le=65535)  # validated automatically
```

FastAPI raises HTTP 422 automatically if validation fails — no manual checking needed.

---

The cells below write the full API to disk. Run the terminal command after to test it.

open a terminal and run:

  "uvicorn lab2_main:app --reload --port 8000"

  
Then visit: http://localhost:8000/docs